# Bartons indicator expressions

Bartons indicators return native Polars expressions. This tutorial uses the bundled daily sample prices to demonstrate selection, composition, multi-output structs, and lazy queries.

In [1]:
import polars as pl

from bartons.indicators import ATR, DMI, EMA, MACD, MOM, RSI, SMA, WILLR
from bartons.samples import sample_prices

## Load the sample prices

`sample_prices` ships with Bartons, so the notebook requires no network access.

In [2]:
prices = sample_prices("daily")
prices.head()

date,open,high,low,close,volume
date,f64,f64,f64,f64,i64
1980-12-12,0.099058,0.099488,0.099058,0.099058,469033600
1980-12-15,0.09432,0.09432,0.09389,0.09389,175884800
1980-12-16,0.087429,0.087429,0.086999,0.086999,105728000
1980-12-17,0.089152,0.089582,0.089152,0.089152,86441600
1980-12-18,0.091737,0.092167,0.091737,0.091737,73449600


## Select indicators

Factories such as `EMA(20)` and `RSI(14)` produce expressions. Their default output names are the lowercase indicator names; use `alias` when selecting the same indicator more than once.

In [3]:
signals = prices.select(
    "date",
    "close",
    EMA(20),
    SMA(20),
    RSI(14),
    ATR(14),
    MOM(10),
    WILLR(14),
)
signals.tail()

date,close,ema,sma,rsi,atr,mom,willr
date,f64,f64,f64,f64,f64,f64,f64
2024-08-05,209.270004,219.648137,223.7965,38.022893,6.739317,-14.690002,-62.577547
2024-08-06,207.229996,218.465457,222.724001,36.137478,6.89508,-17.779999,-67.392581
2024-08-07,209.820007,217.642081,221.566001,40.192313,6.920431,-8.719986,-56.513506
2024-08-08,213.309998,217.229501,220.853001,45.237928,6.809686,-4.180008,-45.531787
2024-08-09,216.240005,217.135264,220.138001,49.11892,6.666851,-1.720001,-34.583051


## Compose expressions

Single-source indicators default to `close`. A source can also be supplied explicitly, passed as the first argument, or piped from another expression.

In [4]:
composed = prices.select(
    "date",
    EMA(10).alias("ema_close"),
    EMA(10, src=pl.col("high")).alias("ema_high"),
    pl.col("close").pipe(EMA, 10).pipe(RSI, 14).alias("rsi_of_ema"),
)
composed.tail()

date,ema_close,ema_high,rsi_of_ema
date,f64,f64,f64
2024-08-05,218.552825,222.201271,49.488553
2024-08-06,216.494129,219.981041,42.476714
2024-08-07,215.280652,218.828124,38.971621
2024-08-08,214.922351,217.986646,37.975185
2024-08-09,215.161925,217.767256,39.096478


## Multi-output indicators

Multi-output indicators are native struct expressions. Keep the structs as columns during the query, then unnest the result when top-level fields are useful.

In [5]:
multi = prices.select("date", "close", MACD(), DMI())
multi.tail()

date,close,macd,dmi
date,f64,struct[3],struct[3]
2024-08-05,209.270004,"{0.279068,2.417884,-2.138816}","{27.150619,14.432837,38.54906}"
2024-08-06,207.229996,"{-0.689506,1.796406,-2.485912}","{28.462564,13.099164,34.986918}"
2024-08-07,209.820007,"{-1.233892,1.190346,-2.424238}","{28.869323,15.886258,32.368841}"
2024-08-08,213.309998,"{-1.36794,0.678689,-2.046629}","{29.124992,15.578824,30.545592}"
2024-08-09,216.240005,"{-1.223642,0.298223,-1.521865}","{28.800149,17.540192,28.971448}"


In [6]:
multi.unnest("macd", "dmi").tail()

date,close,macd,macdsignal,macdhist,adx,pdi,mdi
date,f64,f64,f64,f64,f64,f64,f64
2024-08-05,209.270004,0.279068,2.417884,-2.138816,27.150619,14.432837,38.54906
2024-08-06,207.229996,-0.689506,1.796406,-2.485912,28.462564,13.099164,34.986918
2024-08-07,209.820007,-1.233892,1.190346,-2.424238,28.869323,15.886258,32.368841
2024-08-08,213.309998,-1.36794,0.678689,-2.046629,29.124992,15.578824,30.545592
2024-08-09,216.240005,-1.223642,0.298223,-1.521865,28.800149,17.540192,28.971448


## Lazy queries

The same indicator expressions work in lazy pipelines and can be referenced by later query stages.

In [7]:
overbought = (
    prices.lazy()
    .with_columns(RSI(14), EMA(20))
    .filter(pl.col("rsi") > 70)
    .select("date", "close", "ema", "rsi")
    .collect()
)
overbought.tail()

date,close,ema,rsi
date,f64,f64,f64
2024-07-09,228.679993,214.09955,76.161203
2024-07-10,232.979996,215.897687,78.768501
2024-07-12,230.539993,218.297971,70.828251
2024-07-15,234.399994,219.831497,73.461582
2024-07-16,234.820007,221.258974,73.739367
